In [ ]:
import pandas as pd

df = pd.read_csv(
    r"C:\Users\nando\Desktop\spotify-sentiment-analysis\dataset\reviews.csv"
)

# Visualizar as primeiras linhas
display(df.head())

# Verificar quantidade de linhas e colunas
print(df.shape)

# Verificar os tipos de dados
print(df.info())

# Verificar valores ausentes
print(df.isnull().sum())

# Verificar registros duplicados
print(df.duplicated().sum())

## Distribuição das notas
## 1 e 2 - Negativo
## 3 - Neutro
## 4 e 5 - Positivo

In [ ]:
print(df["Rating"].value_counts().sort_index())

In [ ]:
print(
    df["Rating"].value_counts(normalize=True)
    .sort_index()
    .mul(100)
)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 5))

sns.countplot(data=df, x="Rating")

plt.title("Distribuição das notas das avaliações")
plt.xlabel("Nota")
plt.ylabel("Quantidade")
plt.show()

## Distribuição dos sentimentos

In [18]:
def classify_sentiment(rating):
    if rating <= 2:
        return "Negativo"
    elif rating == 3:
        return "Neutro"
    else:
        return "Positivo"

df["Sentiment"] = df["Rating"].apply(classify_sentiment)

print(df["Sentiment"].value_counts())

Sentiment
Positivo    29937
Negativo    24771
Neutro       6886
Name: count, dtype: int64


In [ ]:
plt.figure(figsize=(8, 5))

sns.countplot(
    data=df,
    x="Sentiment",
    order=["Negativo", "Neutro", "Positivo"]
)

plt.title("Distribuição dos sentimentos")
plt.xlabel("Sentimento")
plt.ylabel("Quantidade")
plt.show()

## Visualizar avaliações de cada sentimento

In [ ]:
for sentiment in ["Positivo", "Negativo", "Neutro"]:
    print(f"\n--- {sentiment} ---")

    reviews = df[df["Sentiment"] == sentiment]["Review"]

    for review in reviews.head(3):
        print("-", review)

## Etapa 1 — Verificar o tamanho das avaliações

In [20]:
df["Review_length"] = df["Review"].apply(len)

print(df["Review_length"].describe())

count    61594.000000
mean       163.323457
std        119.940997
min         10.000000
25%         72.000000
50%        130.000000
75%        221.000000
max       3753.000000
Name: Review_length, dtype: float64


## Etapa 2 — Verificar avaliações vazias ou muito curtas

In [ ]:
print("Avaliações vazias:", (df["Review"].str.strip() == "").sum())

print("\nAvaliações com menos de 5 caracteres:")
print((df["Review_length"] < 5).sum())

## Etapa 3 — Visualizar a distribuição do tamanho dos textos

In [ ]:
plt.figure(figsize=(10, 5))

sns.histplot(
    data=df,
    x="Review_length",
    bins=50
)

plt.title("Distribuição do tamanho das avaliações")
plt.xlabel("Quantidade de caracteres")
plt.ylabel("Frequência")

plt.show()

## Implementação do algoritmo TF-IDF

## Passo 1 — Importar a biblioteca

In [23]:
from sklearn.feature_extraction.text import TfidfVectorizer

## Passo 2 — Preparar os dados

In [24]:
X = df["Review"].fillna("")
y = df["Rating"]

## Passo 3 — Aplicar o TF-IDF

In [25]:
tfidf = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    max_features=5000
)

X_tfidf = tfidf.fit_transform(X)

## Passo 4 — Visualizar o resultado

In [26]:
print(X_tfidf.shape)

import pandas as pd

features = tfidf.get_feature_names_out()

tfidf_df = pd.DataFrame(
    X_tfidf.toarray(),
    columns=features
)

print(tfidf_df.head())

(61594, 5000)
    00  000   01   03   04   05   06   07   08   09  ...  youth  youtube  \
0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  ...    0.0      0.0   
1  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  ...    0.0      0.0   
2  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  ...    0.0      0.0   
3  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  ...    0.0      0.0   
4  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  ...    0.0      0.0   

   youve  ypu   yr  yrs   yt  yup  zero  zone  
0    0.0  0.0  0.0  0.0  0.0  0.0   0.0   0.0  
1    0.0  0.0  0.0  0.0  0.0  0.0   0.0   0.0  
2    0.0  0.0  0.0  0.0  0.0  0.0   0.0   0.0  
3    0.0  0.0  0.0  0.0  0.0  0.0   0.0   0.0  
4    0.0  0.0  0.0  0.0  0.0  0.0   0.0   0.0  

[5 rows x 5000 columns]


## 3. Integrando TF-IDF com Naive Bayes

In [29]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report

# Separação das variáveis
X = df["Review"].fillna("")
y = df["Rating"]

# Divisão dos dados
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Pipeline: TF-IDF + Naive Bayes
modelo = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        stop_words="english",
        max_features=5000
    )),
    ("naive_bayes", MultinomialNB())
])

# Treinamento
modelo.fit(X_train, y_train)

# Previsões
y_pred = modelo.predict(X_test)

# Avaliação
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           1       0.51      0.91      0.65      3531
           2       0.25      0.00      0.01      1424
           3       0.31      0.02      0.03      1377
           4       0.42      0.08      0.14      1568
           5       0.70      0.90      0.79      4419

    accuracy                           0.60     12319
   macro avg       0.44      0.38      0.32     12319
weighted avg       0.52      0.60      0.49     12319



In [31]:
def classificar_sentimento(rating):
    if rating <= 2:
        return "Negativo"
    elif rating == 3:
        return "Neutro"
    else:
        return "Positivo"

df["Sentiment"] = df["Rating"].apply(classificar_sentimento)

print(df["Sentiment"].value_counts())

Sentiment
Positivo    29937
Negativo    24771
Neutro       6886
Name: count, dtype: int64


In [32]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB

def criar_modelo():
    return Pipeline([
        ("tfidf", TfidfVectorizer(
            lowercase=True,
            stop_words="english",
            max_features=5000
        )),
        ("naive_bayes", MultinomialNB())
    ])

In [33]:
from sklearn.model_selection import train_test_split

indices = df.index

idx_train, idx_test = train_test_split(
    indices,
    test_size=0.2,
    random_state=42,
    stratify=df["Rating"]
)

X_train = df.loc[idx_train, "Review"].fillna("")
X_test = df.loc[idx_test, "Review"].fillna("")

In [34]:
from sklearn.metrics import classification_report, accuracy_score

y_train_rating = df.loc[idx_train, "Rating"]
y_test_rating = df.loc[idx_test, "Rating"]

modelo_rating = criar_modelo()

modelo_rating.fit(X_train, y_train_rating)

pred_rating = modelo_rating.predict(X_test)

print("CLASSIFICAÇÃO DE RATINGS (1 A 5)")
print(classification_report(y_test_rating, pred_rating))

print("Accuracy:", accuracy_score(y_test_rating, pred_rating))

CLASSIFICAÇÃO DE RATINGS (1 A 5)
              precision    recall  f1-score   support

           1       0.51      0.91      0.65      3531
           2       0.25      0.00      0.01      1424
           3       0.31      0.02      0.03      1377
           4       0.42      0.08      0.14      1568
           5       0.70      0.90      0.79      4419

    accuracy                           0.60     12319
   macro avg       0.44      0.38      0.32     12319
weighted avg       0.52      0.60      0.49     12319

Accuracy: 0.595502881727413


In [35]:
y_train_sentiment = df.loc[idx_train, "Sentiment"]
y_test_sentiment = df.loc[idx_test, "Sentiment"]

modelo_sentiment = criar_modelo()

modelo_sentiment.fit(X_train, y_train_sentiment)

pred_sentiment = modelo_sentiment.predict(X_test)

print("CLASSIFICAÇÃO DE SENTIMENTOS")
print(classification_report(y_test_sentiment, pred_sentiment))

print("Accuracy:", accuracy_score(y_test_sentiment, pred_sentiment))

CLASSIFICAÇÃO DE SENTIMENTOS
              precision    recall  f1-score   support

    Negativo       0.71      0.87      0.78      4955
      Neutro       0.14      0.00      0.00      1377
    Positivo       0.82      0.87      0.84      5987

    accuracy                           0.77     12319
   macro avg       0.56      0.58      0.54     12319
weighted avg       0.70      0.77      0.73     12319

Accuracy: 0.769461806964851


In [ ]:
from sklearn.metrics import f1_score, accuracy_score
import pandas as pd

resultados = pd.DataFrame({
    "Métrica": ["Accuracy", "F1 Macro", "F1 Weighted"],
    "Ratings (1-5)": [
        accuracy_score(y_test_rating, pred_rating),
        f1_score(y_test_rating, pred_rating, average="macro"),
        f1_score(y_test_rating, pred_rating, average="weighted")
    ],
    "Sentimentos (3 classes)": [
        accuracy_score(y_test_sentiment, pred_sentiment),
        f1_score(y_test_sentiment, pred_sentiment, average="macro"),
        f1_score(y_test_sentiment, pred_sentiment, average="weighted")
    ]
})

print(resultados.to_string(index=False))

In [39]:
print(classification_report(
    y_test_sentiment,
    pred_sentiment
))

              precision    recall  f1-score   support

    Negativo       0.71      0.87      0.78      4955
      Neutro       0.14      0.00      0.00      1377
    Positivo       0.82      0.87      0.84      5987

    accuracy                           0.77     12319
   macro avg       0.56      0.58      0.54     12319
weighted avg       0.70      0.77      0.73     12319



In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
import matplotlib.pyplot as plt

ConfusionMatrixDisplay.from_predictions(
    y_test_sentiment,
    pred_sentiment,
    cmap="Blues"
)

plt.title("Matriz de Confusão - Análise de Sentimentos")
plt.show()

In [42]:
modelo_sentiment = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        stop_words="english",
        max_features=5000
    )),
    ("naive_bayes", MultinomialNB(
        fit_prior=False
    ))
])

modelo_sentiment.fit(X_train, y_train_sentiment)

pred_sentiment = modelo_sentiment.predict(X_test)

print(classification_report(
    y_test_sentiment,
    pred_sentiment
))

              precision    recall  f1-score   support

    Negativo       0.75      0.76      0.75      4955
      Neutro       0.22      0.32      0.26      1377
    Positivo       0.88      0.79      0.83      5987

    accuracy                           0.72     12319
   macro avg       0.62      0.62      0.62     12319
weighted avg       0.75      0.72      0.74     12319



In [ ]:
from sklearn.naive_bayes import ComplementNB

modelo_complement = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        stop_words="english",
        max_features=5000
    )),
    ("naive_bayes", ComplementNB(alpha=1.0))
])

modelo_complement.fit(X_train, y_train_sentiment)

pred_complement = modelo_complement.predict(X_test)

print(classification_report(
    y_test_sentiment,
    pred_complement
))

In [ ]:
from sklearn.metrics import f1_score, accuracy_score
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer

valores_alpha = [0.01, 0.1, 0.5, 1.0, 2.0]

resultados_alpha = []

for alpha in valores_alpha:

    modelo = Pipeline([
        ("tfidf", TfidfVectorizer(
            lowercase=True,
            stop_words="english",
            max_features=5000
        )),
        ("naive_bayes", MultinomialNB(
            alpha=alpha,
            fit_prior=False
        ))
    ])

    modelo.fit(X_train, y_train_sentiment)

    pred = modelo.predict(X_test)

    resultados_alpha.append({
        "Alpha": alpha,
        "Accuracy": accuracy_score(y_test_sentiment, pred),
        "F1 Macro": f1_score(
            y_test_sentiment, pred, average="macro"
        ),
        "F1 Weighted": f1_score(
            y_test_sentiment, pred, average="weighted"
        ),
        "F1 Neutro": f1_score(
            y_test_sentiment,
            pred,
            labels=["Neutro"],
            average="macro",
            zero_division=0
        )
    })

df_resultados_alpha = pd.DataFrame(resultados_alpha)

print(df_resultados_alpha.to_string(index=False))